In [1]:
import re, os, time, json, math
from functools import lru_cache
from urllib.parse import quote_plus

import requests
import gradio as gr

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Group7-TravelChatbot/1.0 (HF Spaces educational)"})

def _get(url, params=None, timeout=12):
    for i in range(3):
        try:
            r = SESSION.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r.json() if "application/json" in r.headers.get("Content-Type", "") else r.text
        except requests.RequestException:
            pass
        time.sleep(0.6 * (i + 1))
    return None

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

@lru_cache(maxsize=512)
def geocode_place(query: str):
    if not query or not query.strip():
        return None
    params = {
        "q": query.strip(),
        "format": "jsonv2",
        "addressdetails": 1,
        "limit": 1,
        "accept-language": "en",
    }
    data = _get(NOMINATIM_URL, params=params)
    if not data or not isinstance(data, list) or len(data) == 0:
        return None
    item = data[0]
    addr = item.get("address", {})
    try:
        return {
            "name": item.get("name") or query.strip(),
            "display_name": item.get("display_name"),
            "lat": float(item.get("lat")),
            "lon": float(item.get("lon")),
            "country": addr.get("country"),
            "country_code": addr.get("country_code"),
        }
    except Exception:
        return None

def _wiki_rest_summary(title: str):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote_plus(title)}"
    try:
        r = SESSION.get(url, timeout=10)
        if r.status_code != 200:
            return None
        data = r.json()
        if data.get("type") == "disambiguation":
            return None
        extract = data.get("extract") or ""
        if not extract.strip():
            return None
        return {
            "title": data.get("title", title),
            "summary": extract.strip(),
            "url": data.get("content_urls", {}).get("desktop", {}).get("page",
                   f"https://en.wikipedia.org/wiki/{quote_plus(title)}"),
        }
    except Exception:
        return None

def _wiki_search_title(query: str):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "opensearch",
        "search": query,
        "limit": 1,
        "namespace": 0,
        "format": "json",
    }
    try:
        r = SESSION.get(url, params=params, timeout=10)
        if r.status_code != 200:
            return None
        data = r.json()
        if isinstance(data, list) and len(data) >= 2 and data[1]:
            return data[1][0]
    except Exception:
        return None
    return None

def get_wiki_summary(place: str):
    if not place:
        return None
    info = _wiki_rest_summary(place)
    if info:
        return info
    title = _wiki_search_title(place)
    if title:
        return _wiki_rest_summary(title)
    return None

NON_ASCII = re.compile(r"[^\x00-\x7F]")
try:
    from deep_translator import GoogleTranslator
    _translator = GoogleTranslator(source='auto', target='en')
except Exception:
    _translator = None

def to_english(text: str) -> str:
    if not text:
        return text
    g = geocode_place(text)
    name = (g.get("display_name") or g.get("name") or text) if g else text
    name = name.split(",")[0].strip()
    if NON_ASCII.search(name) and _translator:
        try:
            name = _translator.translate(name)
        except Exception:
            pass
    return name

STOPWORDS = set("""
a an the and or for of on at to in from with by about around near what where when how why can could should would do does did is are am
things thing see visit go best top list tell me
""".split())

def candidate_spans(text):
    t = text.strip()
    m = re.search(r"\b(in|at|to|near|around)\s+([A-Z][A-Za-zÀ-ÿ' -]{1,60})", t)
    if m:
        yield m.group(2).strip()
    caps = re.findall(r"(?:[A-Z][A-Za-zÀ-ÿ']+(?:\s+[A-Z][A-Za-zÀ-ÿ']+)*)", t)
    for span in caps:
        if span.lower() not in STOPWORDS and len(span) > 1:
            yield span.strip()
    words = re.findall(r"[A-Za-zÀ-ÿ']+", t)
    for n in (3, 2, 1):
        for i in range(len(words) - n + 1):
            span = " ".join(words[i:i+n]).strip()
            if span.lower() in STOPWORDS:
                continue
            if any(w.lower() in STOPWORDS for w in span.split()):
                continue
            if any(w[0].isupper() for w in span.split()):
                yield span

def resolve_place(text):
    for span in candidate_spans(text):
        g = geocode_place(span)
        if g:
            return g
    return geocode_place(text)

def robust_place_or_title(text: str):
    text = (text or "").strip()
    loc = resolve_place(text)
    if loc:
        title = to_english(loc.get("name"))
        return title, loc.get("country")
    title = _wiki_search_title(text)
    if title:
        return to_english(title), None
    caps = re.findall(r"[A-Z][A-Za-zÀ-ÿ'-]+", text)
    cand = caps[-1] if caps else (re.findall(r"[A-Za-zÀ-ÿ'-]+", text)[-1] if re.findall(r"[A-Za-zÀ-ÿ'-]+", text) else None)
    return (to_english(cand), None) if cand else (None, None)


DATE_PAT = re.compile(r"\b(\d{4}-\d{2}-\d{2}|\d{1,2}/\d{1,2}/\d{2,4})\b")

def _normalize_date(s):
    s = s.strip()
    if re.match(r"\d{4}-\d{2}-\d{2}$", s):
        return s
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{2,4})$", s)
    if m:
        mm, dd, yy = m.groups()
        mm = mm.zfill(2); dd = dd.zfill(2)
        if len(yy) == 2:
            yy = ("20" if int(yy) <= 68 else "19") + yy
        return f"{yy}-{mm}-{dd}"
    return None

def _parse_dates_with_context(text: str):
    depart = ret = None
    for m in re.finditer(DATE_PAT, text):
        raw = m.group(1)
        d = _normalize_date(raw)
        if not d:
            continue
        window = text[max(0, m.start()-20): m.end()+20].lower()
        left   = text[max(0, m.start()-15): m.start()].lower()

        if re.search(r"\b(return|returning|back)\b", window):
            ret = d
        elif re.search(r"\b(depart|departure|leave|leaving|outbound)\b", window) or re.search(r"\bon\b$", left):
            if not depart:
                depart = d
        else:
            if not depart:
                depart = d
            elif not ret:
                ret = d
    try:
        if depart and ret and depart > ret:
            depart, ret = ret, depart
    except Exception:
        pass
    return depart, ret

def normalize_flight_text(text: str) -> str:
    t = text.lower().strip()
    t = re.sub(r"\bi\s*(need|want|am looking for|would like|plan|book)\b", "flights", t)
    t = t.replace("flight", "flights")
    t = re.sub(r"\b(please|kindly|can you|show me|help me|get me|to book)\b", "", t)
    return " ".join(t.split())

def extract_flight_query(text: str):
    if not text or ("flight" not in text.lower() and "flights" not in text.lower()):
        return None

    depart, ret = _parse_dates_with_context(text)

    cleaned = text
    for m in re.findall(DATE_PAT, text):
        raw = m if isinstance(m, str) else m[0]
        cleaned = re.sub(rf"\b(on|returning|depart(?:ure)?|leaving|back)\s*{re.escape(raw)}\b",
                         "", cleaned, flags=re.IGNORECASE)
        cleaned = cleaned.replace(raw, "")
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()

    origin = destination = None
    m = re.search(r"\bfrom\s+([A-Za-zÀ-ÿ' \-]{2,60})\s+to\s+([A-Za-zÀ-ÿ' \-]{2,60})\b", cleaned, re.IGNORECASE)
    if m:
        origin = m.group(1).strip(" ,.")
        destination = m.group(2).strip(" ,.")
    else:
        m = re.search(r"\bto\s+([A-Za-zÀ-ÿ' \-]{2,60})\s+from\s+([A-Za-zÀ-ÿ' \-]{2,60})\b", cleaned, re.IGNORECASE)
        if m:
            destination = m.group(1).strip(" ,.")
            origin = m.group(2).strip(" ,.")

    def to_english_name(q):
        if not q: return None
        return to_english(q)

    origin = to_english_name(origin)
    destination = to_english_name(destination)

    return {"origin": origin, "destination": destination, "depart": depart, "return": ret}

def flights_link(q):
    def clean_for_url(s):
        if not s:
            return ""
        s_ascii = s.encode("ascii", errors="ignore").decode()
        return quote_plus(s_ascii.strip())

    parts = []
    if q.get("origin"): parts.append(f"from {clean_for_url(q['origin'])}")
    if q.get("destination"): parts.append(f"to {clean_for_url(q['destination'])}")
    if q.get("depart") and q.get("return"):
        parts.append(f"{q['depart']} to {q['return']}")
    elif q.get("depart"):
        parts.append(f"on {q['depart']}")
    base_query = "flights " + " ".join(parts) if parts else "flights"
    return f"https://www.google.com/travel/flights?q={quote_plus(base_query)}"


SAFETY_KEYWORDS = re.compile(
    r"\b(safe|safety|unsafe|crime|danger|dangerous|pickpocket|scam|scams|scammers|precaution|precautions|risk|risky|emergency|security|is it safe)\b",
    re.IGNORECASE
)

SAFETY_TIPS_GENERIC = [
    "Keep valuables out of sight and use a money belt or zipped pockets in crowded places.",
    "Avoid poorly lit or deserted areas at night; stick to well-known streets and neighborhoods.",
    "Use registered taxis or reputable ride-hailing apps; confirm vehicle/driver details before entering.",
    "Beware of common scams (fake petitions, 'friendship bracelets', unofficial tour helpers). Politely decline and walk away.",
]

SAFETY_TIPS_BY_PLACE = {
    "paris": [
        "High-traffic areas (Eiffel Tower, Montmartre, major metro stations) attract pickpockets—secure bags and phones.",
        "On the metro, keep backpacks in front; avoid holding your phone near train doors."
    ],
    "nairobi": [
        "Use registered taxis or trusted ride-hailing; avoid walking alone late at night in unfamiliar areas.",
        "Leave passports in a hotel safe; carry a photocopy or digital copy when out."
    ],
    "cairo": [
        "Expect persistent vendors near major sights; a firm, polite 'no thank you' usually works.",
        "Drink sealed bottled water; be cautious with street food if you have a sensitive stomach."
    ],
    "tokyo": [
        "Trains are very safe; still, avoid blocking doors and keep phones secure in crowded cars.",
        "Follow local etiquette (quiet on trains, proper trash sorting); lost items are often returned to station staff."
    ],
}

def detect_safety_intent(text: str) -> bool:
    return bool(SAFETY_KEYWORDS.search(text or ""))

def safety_message(location_info: dict | None) -> str:
    place_name = location_info.get("name") if location_info else None
    place_key = (place_name or "").lower()
    country = location_info.get("country") if location_info else None

    bullets = []
    if place_key in SAFETY_TIPS_BY_PLACE:
        bullets.extend(SAFETY_TIPS_BY_PLACE[place_key])
    bullets.extend(SAFETY_TIPS_GENERIC[:4])

    header = f"🛡️ Safety tips{f' for **{to_english(place_name)}**' if place_name else ''}{f' — {country}' if country else ''}"
    body = "\n".join([f"- {b}" for b in bullets])
    footer = "\n\nℹ️ These are general reminders. For official guidance, consult local authorities, your embassy/consulate, and recent traveler advisories."
    return f"{header}\n\n{body}{footer}"


from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

LM_NAME = "distilgpt2"
_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
_model = AutoModelForCausalLM.from_pretrained(LM_NAME)
_chatbot = pipeline("text-generation", model=_model, tokenizer=_tokenizer, device=-1)

def travel_chatbot(user_input):
    try:
        text = (user_input or "").strip()
        if not text:
            return "Please type a destination or travel question."

        norm_text = normalize_flight_text(text)

        if detect_safety_intent(norm_text):
            loc = resolve_place(norm_text)
            return safety_message(loc)

        fq = extract_flight_query(norm_text)
        if fq and (fq["origin"] or fq["destination"]):
            link = flights_link(fq)
            pretty = "Flights"
            if fq["origin"] and fq["destination"]:
                pretty += f" from **{fq['origin']}** to **{fq['destination']}**"
            if fq["depart"]:
                pretty += f" on **{fq['depart']}**"
            if fq["return"]:
                pretty += f" returning **{fq['return']}**"
            return f"✈️ {pretty}\n\nSearch on Google Flights: {link}"

        title, country = robust_place_or_title(norm_text)
        if title:
            wiki = get_wiki_summary(title)
            if wiki:
                return (
                    f"🌍 **{wiki['title']}**" + (f" — {country}" if country else "") + "\n\n"
                    f"{wiki['summary']}\n\n"
                    f"📖 Read more: {wiki['url']}"
                )

        out = _chatbot(norm_text, max_new_tokens=60, do_sample=True, temperature=0.8)
        return out[0]["generated_text"]

    except Exception as e:
        return f"⚠️ Error: {e}"

def respond(message, history):
    return travel_chatbot(message)

demo = gr.ChatInterface(
    fn=respond,
    title="Group 7 Travel Guidance Chatbot",
    description=(
        "Ask about any city/country, get safety reminders, or build a Google Flights link.\n"
        "Examples: 'Tell me about Cairo' · 'Is it safe to travel to Nairobi?' · "
        "'I need a flight from Rome to Tokyo on 12/12/2025'."
    ),
    theme="soft",
)

if __name__ == "__main__":
    demo.launch()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d2e47e12c9d3a60c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
